In [1]:
import numpy as np
import pandas as pd
import phonopy
import matplotlib.pyplot as plt
import ase.io
from ase import Atoms

In [2]:
# This notebook is to run electron phonon coupling with 
# singular value decomposition (SVD), which can provide different
# perspective for designing high mobility of OSCs
print("Note: Dimer is formed by 2 monomers")

Note: Dimer is formed by 2 monomers


In [3]:
### Define units for each variable ###
mass_au_to_kg = 1.6605e-27 # convert mass from a.u. into kg -> MASS: 1.6605e-27 kg
boltzmann_ev = 8.617e-5 # Boltzmann constant: 8.617e-5 eV/K
boltzmann_J = 1.381e-23 # Boltzmann constant in 1.381e-23 J⋅K−1
planck_J = 6.626e-34 # Planck constant in 6.626e-34 J⋅s
planck_ev = 4.136e-15 # planck constant in 4.1357e−15 eV⋅s;
reduced_planck_J = 1.055e-34 # Reduced constant in J⋅s
thz_to_wnumber = 33.356 # FREQUENCY: 1 THz = 33.356 cm^-1. In phononpy is THz.

In [4]:
def get_deri_Jmatrix(j_list, delta=0.01):
    """ Calculate derivative of transfer integral J and return as electron-phonon coupling matrix 
    Args:
        j_list (list): list of transfer integrals for each dimer 
        delta (float): size of displacement (Defaults to 0.01 Angstrom)
    Returns:
        matrix: matrix containing gradient of J_ij
    """
    num_atoms = len(j_list) // 6 # 6 because each atom have 6 displaced treatment, 3 directions (x,y,z) and 2 sign (+ & -)
    
    # array with j- (negative displacement)
    j_n = np.empty([num_atoms, 3]) 
    j_n[:, 0] = j_list[0::6] # atom x -
    j_n[:, 1] = j_list[2::6] # atom y -
    j_n[:, 2] = j_list[4::6] # atom z -
    
    # array with j+ (positive displacement)
    j_p = np.empty([num_atoms, 3])
    j_p[:, 0] = j_list[1::6] # atom x +
    j_p[:, 1] = j_list[3::6] # atom y +
    j_p[:, 2] = j_list[5::6] # atom z +

    matrix = (np.abs(j_p) - np.abs(j_n)) / (2 * delta) # The sign in J depends on an arbitrary sign of the basis orbital.
    return matrix

In [5]:
def ucell_info(crystal, monomer1, monomer2):
    """ Obtain some crystal structure (unitcell) info, would ask the user to provide translation vectors
    Args:
        crystal: crystal structure file
        monomer1 (int): label for monomer 1
        monomer2 (int): label for monomer 2
    Return:
        atoms_ucell: ASE atoms for the unitcell
        atomic_mass: Atomic mass for each atom in unitcell
        n_atoms_ucell: Number of atoms in unitcell
        n_modes (int) : Total number of phonon modes
        t_vecs (list): Translation vectors
        translation_dimer (y/n): Decide whether this dimer is formed by 2 translation molecules or not
    """
    atoms_ucell = ase.io.read(crystal) # Real the unitcell
    n_atoms_ucell = len(atoms_ucell) # number of atoms
    n_modes = 3 * n_atoms_ucell # Number of modes (3N)
    atomic_mass = atoms_ucell.get_masses() # Atomic mass for each atom in unitcell
    com = np.load('../center_of_mass.npz')['arr_0'] # center of mass
    cell = atoms_ucell.get_cell() # cell vectors
    distance = com[monomer2-1] - com[monomer1-1] # distance between center of mass for 2 monomers (distance = ax + by + cz)
    frac = np.linalg.solve(cell.T, distance) # obtain the coefficient 
    dist_vectors = np.abs(np.round(frac, 4)) # this is distance vectors based on the fraction of cell vectors
    print(f"--- Lattice translational vectors: {dist_vectors}\n"
          "If the index not the integer, it is still in the unitcell. It should be 0 0 0\n"
          "Please define translation vectors by yourself!!! ---")
    vecs_input = input("Enter numbers separated by space: ")
    t_vecs = list(map(int, vecs_input.split()))
    print("--- If the translation vectors is not zero instead is an integer at any direction, select y!!! ---")
    translation_dimer = input("This dimer is formed by tranlsation molecules (y or n)").strip().lower()

    return atoms_ucell, atomic_mass, n_atoms_ucell, n_modes, t_vecs, translation_dimer

In [6]:
def phonon():
    """ Call phonopy to solve hessian matrix and obtain frequencies and eigenvectors
    Return:
        eigenvecs (array): eigenvectors
        freqs (array): Frequency array in shape of (q, I)
        freqs_flatten (array): Frequency array in shape of Iq (mode index at each qpts) 
        n_qpts (int): Number of qpts
        qpts (array): qpts coordinates 
    """
    print("--- Solving Hessian Matrix (second order derivative of potential energy) by phonopy ---")
    phonon = phonopy.load('phonopy_disp.yaml')
    phonon.run_mesh(mesh=mesh,with_eigenvectors=True)
    mesh_data = phonon.get_mesh_dict() # Solve the hessian matrix
    eigenvecs = mesh_data['eigenvectors']
    freqs = mesh_data['frequencies'] # Shape is (nqpts*natoms*3,)
    qpts = mesh_data['qpoints'] 

    print("--- Check whether there is any negative frequencies and remove them ---")
    negative_mask = (freqs < 0).any(axis=1)  # Remove q-points with negative frequency
    valid_mask = ~negative_mask
    n_removed = negative_mask.sum()
    print(f'--- Removing {n_removed} q-points containing imaginary modes ---')
    freqs = freqs[valid_mask]
    eigenvecs = eigenvecs[valid_mask]
    qpts = qpts[valid_mask]
    n_qpts = len(qpts)
    freqs_flatten = freqs.T.flatten()

    print(f'Filtered frequency shape: {freqs.shape}')
    print(f'Number of remaining q-points = {n_qpts}')
    print(f'Eigenvectors shape (qpts, dof, mode) = {eigenvecs.shape}')
    print('--- Finishing Solving Hessian Matrix ---')
    return eigenvecs, freqs, freqs_flatten, n_qpts, qpts

In [7]:
def mapping_atoms(dimer_label, translation_dimer, xyz):
    """ Map the dimers by reading mapping file, the atomic order need to follow crystal file
    Args: 
        dimer_label (str): a, b, c ...
        xyz: The dimer structure file
        translation_dimer (y/n): Need special mapping if dimer is translation molecules
    Return:
        dimer_map file: The .xyz file after mapping, the atomic index is in order of the atomic index in crystal file
        inverse_mapping: The mapping for crystal to dimer (c1 -> d#, c2 -> d# ...), this will be used to identify an atom is in dimer 1 or 2. This is to identify translation 
        for phase factor
    """
    mapping = np.load(f"../mapping/map_{dimer_label.upper()}.npz")["mapping"] # Dimer to crystal (d1 -> c#, d2 -> c# ...)
    print(f'--- Atomic index mapping (dimer # : crystal #): Ex: d1 -> n; d2 -> m ...\n'
          f'{mapping} ---')
    atoms_xyz = ase.io.read(xyz)
    n_atoms_xyz = len(atoms_xyz)

    if translation_dimer == 'y': # 2 scenario. If 2 mols in unitcell. Only 1 will be matched. Or 1 mol in unitcell and need to write explicilty
        half_n_atoms_xyz = int(n_atoms_xyz / 2) # the number of atoms for the monomer (half dimer)
        inverse_mapping1 = list(np.argsort(mapping[0:half_n_atoms_xyz])) # Crystal to dimer (c1 -> d#, c2 -> d# ...)
        inverse_mapping2 = np.argsort(mapping[half_n_atoms_xyz:]) # Crystal to dimer (c1 -> d#, c2 -> d# ...)
        inverse_mapping2 = [mp + half_n_atoms_xyz for mp in inverse_mapping2] # Make the index match 2nd monomer by adding the number of atoms im 1 monomer
        inverse_mapping = inverse_mapping1 + inverse_mapping2
        atoms_new_map1 = [None] * (n_atoms_xyz)
        atoms_new_map2 = [None] * (n_atoms_xyz)
        for i in range(0, half_n_atoms_xyz): # i is the index from xyz file
            i_map = mapping[i] # i_map would be the index match the unitcell (i : i_map)
            atoms_new_map1[i_map] = atoms_xyz[i]
        for i in range(half_n_atoms_xyz, n_atoms_xyz): # i is the index from xyz file
            i_map = mapping[i] # i_map would be the index match the unitcell (i : i_map)
            atoms_new_map2[i_map] = atoms_xyz[i]
        atoms_new_map1 = [atom for atom in atoms_new_map1 if atom is not None] # Atoms are in order of unitcell 
        atoms_new_map2 = [atom for atom in atoms_new_map2 if atom is not None] # Atoms are in order of unitcell 
        atoms_map_xyz1 = Atoms(atoms_new_map1)
        atoms_map_xyz2 = Atoms(atoms_new_map2)
        atoms_map_combined = atoms_map_xyz1 + atoms_map_xyz2 # Order: (monomer1, monomer2). Each monomer is in order of unitcell
        ase.io.write(f'dimer_{dimer_label.upper()}_map.xyz', atoms_map_combined, format='xyz')
    else:
        inverse_mapping = np.argsort(mapping) # Crystal to dimer (c1 -> d#, c2 -> d# ...)
        atoms_new_map = [None] * (n_atoms_xyz)
        for i in range(n_atoms_xyz): # i is the index from xyz file
            i_map = mapping[i] # i_map would be the index match the unitcell (i : i_map)
            atoms_new_map[i_map] = atoms_xyz[i]
        atoms_map_xyz = Atoms(atoms_new_map)
        ase.io.write(f'dimer_{dimer_label.upper()}_map.xyz', atoms_map_xyz, format='xyz')

    print('--- Finishing Mapping and Writing dimer_map.xyz file to the folder ---')
    return inverse_mapping

In [8]:
def freq_summary_for_mode(freqs, I):
    """ This function would print the frequency based on the mode I
    Args: 
        freqs: Frequencies array from phonopy
        I (int): The mode index
    Return:
        freq_mode.min(): min freq in wavenumber 
        freq_mode.max(): max freq in wavenumber 
        freq_mode.mean(): mean freq in wavenumber 
    """
    freq_mode = freqs[:, I] * thz_to_wnumber
    return float(freq_mode.min()), float(freq_mode.max()), float(freq_mode.mean())

In [9]:
def write_original_motion(atoms_ucell, atomic_mass, dimer_label, dof, eigenvecs, freqs, inverse_mapping, I, n_atoms_xyz, n_qpts, t_vecs):
    """ Save the original phonon motions, selected the peak from INS spectrum
    Note: If it's translation dimer, the dimer only match to one molecule in the unitcell (in general there are 2 molecules in unitcell)
    In this case, user have to enter the atomic index in the unitcell to match the situation in dimers (ex: 0 2 4 ... 0 2 4 ...)
    Args: 
        atoms_ucell: ASE atoms for the unitcell
        atomic_mass: Atomic mass for each atom in unitcell
        dimer_label (str): a, b, c ...
        dof (int): Degrees of freedom
        eigenvecs (array): eigenvectors
        freqs (array): Frequency array in shape of (q, I)
        inverse_mapping (list):  The mapping for crystal to dimer (c1 -> d#, c2 -> d# ...), this will be used to identify atom is in dimer 1 or 2
        I (int): The mode index
        n_atoms_xyz (int): number of atoms in dimer
        n_qpts (int): Number of qpts
        t_vecs (list): Translation vectors
    Return:
        Save the displacement file to the folder
    """
    n_atoms_ucell = len(atoms_ucell)
    original_disp_matrix = np.zeros((dof, 1), dtype=np.complex128)
    select_atoms_input = input("Enter the index of atoms in the unitcell that will be calculated (Press enter will calculated all atoms (When it's not translation dimer))\n"
                               "If translation dimer, enter the unique unit-cell atom indices corresponding to one monomer (ex: 0 1 2 3 0 1 2 3, enter: 0 1 2 3)").strip()
    if select_atoms_input == "":
        select_atoms = []  # means all atoms
    elif "-" in select_atoms_input: # Ex: 0-3 
        start, end = map(int, select_atoms_input.split("-"))
        base_list = list(range(start, end + 1))
        select_atoms = base_list + base_list  # duplicate for dimer
    else: # Ex: 0 1 2 3
        base_list = list(map(int, select_atoms_input.split()))
        select_atoms = base_list + base_list  # duplicate for dimer

    if len(select_atoms) > 0:
        new_atoms = atoms_ucell[select_atoms]
        new_atomic_mass = new_atoms.get_masses()
        for i, k in enumerate(select_atoms): # loop over atomic index i, k is the index in unitcell 
            am = new_atomic_mass[i] * mass_au_to_kg # atomic mass
            check_map = inverse_mapping[i]
            if check_map < n_atoms_xyz / 2: # dimer 1
                phase = 1 # assume dimer 1 stay in the unitcell, no translation vector
                for ll in (0, 1, 2): # loop over x,y,z directions
                    uq = 0
                    for q in range(n_qpts): # loop over q-points index q
                        freq = freqs[q][I]
                        pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                        eps = eigenvecs[q, k * 3 + ll, I]
                        uq += eps * pref * phase / np.sqrt(n_qpts)
                    original_disp_matrix[i * 3 + ll,] = uq

            elif check_map >= n_atoms_xyz / 2: # dimer 2
                for ll in (0, 1, 2): # loop over x,y,z directions
                    uq = 0
                    for q in range(n_qpts): # loop over q-points index q
                        phase = np.exp(-2j*np.pi * np.dot(qpts[q], t_vecs)) # dimer 2 can be translation of dimer 1 or not
                        freq = freqs[q][I]
                        pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                        eps = eigenvecs[q, k * 3 + ll, I]
                        uq += eps * pref * phase / np.sqrt(n_qpts)
                    original_disp_matrix[i * 3 + ll] = uq

    else:
        for k in range(n_atoms_ucell): # loop over atomic index k
            am = atomic_mass[k] * mass_au_to_kg # atomic mass
            check_map = inverse_mapping[k]
            if check_map < n_atoms_xyz / 2: # dimer 1
                phase = 1 # assume dimer 1 stay in the unitcell, no translation vector
                for ll in (0, 1, 2): # loop over x,y,z directions
                    uq = 0
                    for q in range(n_qpts): # loop over q-points index q
                        freq = freqs[q][I]
                        pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                        eps = eigenvecs[q, k * 3 + ll, I]
                        uq += eps * pref * phase / np.sqrt(n_qpts)
                    original_disp_matrix[k * 3 + ll,] = uq

            elif check_map >= n_atoms_xyz / 2: # dimer 2
                for ll in (0, 1, 2): # loop over x,y,z directions
                    uq = 0
                    for q in range(n_qpts): # loop over q-points index q
                        phase = np.exp(-2j*np.pi * np.dot(qpts[q], t_vecs)) # dimer 2 can be translation of dimer 1 or not
                        freq = freqs[q][I]
                        pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                        eps = eigenvecs[q, k * 3 + ll, I]
                        uq += eps * pref * phase / np.sqrt(n_qpts)
                    original_disp_matrix[k * 3 + ll] = uq

    disp = original_disp_matrix[:,].real
    disp_cart = disp.reshape(-1,3)
    np.savetxt(f"disp/disp_orig_{dimer_label}_m{I}.txt", disp_cart)

In [10]:
def make_dJ_matrix(dimer_label, n_atoms_xyz, translation_dimer):
    """ Create delta J matrix by calling get_deri_J_matrix();
        Map dJ to follow the atomic order in unitcell
    Args:
        dimer_label (str): a, b, c ...
        n_atoms_xyz: Number of atoms in dimer
        translation_dimer (y/n): If y, this needs special treatment of mapping
    Return:
        delJ_matrix: The matrix of numerical derivative of transfer integral J. Shape is (dof, dof)
    """
    print('--- Generate delta J matrix ---')
    jlist_data = np.loadtxt(f"../disp_j/{dimer_label.upper()}.txt", comments="#", dtype=str)
    jlist = jlist_data[:,1].astype(float) # n_atoms_xyz * 6 (6 comes from 3 directions and + - 2 ways)
    delJ = get_deri_Jmatrix(jlist) # This will be delta J data 
    print('--- Obtain delta J matrix, now start to map dJ ---')
    mapping = np.load(f"../mapping/map_{dimer_label.upper()}.npz")["mapping"] # Dimer to crystal (d1 -> c#, d2 -> c# ...)
    if translation_dimer == 'y': # 2 scenario. If 2 mols in unitcell. Only 1 will be matched. Or 1 mol in unitcell and need to write explicilty
        delJ_map_li1 = [None] * n_atoms_xyz
        delJ_map_li2 = [None] * n_atoms_xyz
        half_n_atoms_xyz = int(n_atoms_xyz / 2)
        for i in range(0, half_n_atoms_xyz): # i is the index from xyz file
            i_map = mapping[i] # i_map would be the index match the unitcell (i : i_map)
            delJ_map_li1[i_map] = delJ[i]
        for i in range(half_n_atoms_xyz, n_atoms_xyz):
            i_map = mapping[i]
            delJ_map_li2[i_map] = delJ[i]
        delJ_map_li1 = [j for j in delJ_map_li1 if j is not None]
        delJ_map_li2 = [j for j in delJ_map_li2 if j is not None]
        delJ_map_combined = delJ_map_li1 + delJ_map_li2
        delJ_map = np.array(delJ_map_combined)
        delJ_matrix = np.diag(delJ_map.flatten())
    else:
        delJ_map_li = [None] * n_atoms_xyz
        for i in range(n_atoms_xyz): # i is the index from xyz file
            i_map = mapping[i] # i_map would be the index match the unitcell (i : i_map)
            delJ_map_li[i_map] = delJ[i]
        delJ_map = np.array(delJ_map_li)
        delJ_matrix = np.diag(delJ_map.flatten())
    print('--- Finish generating delta J matrix \n'
          f'Delta J matrix shape: {delJ_matrix.shape} ---')

    return delJ_matrix

In [11]:
def make_XQ_matrix(atoms_ucell, atomic_mass, dof, eigenvecs, freqs, inverse_mapping, n_atoms_xyz, n_modes, n_qpts, t_vecs):
    """ Create atomic displacement X matrix and thermal population Q matrix 
    Note: If it's translation dimer, the dimer only match to one molecule in the unitcell (in general there are 2 molecules in unitcell)
    In this case, user have to enter the atomic index in the unitcell to match the situation in dimers (ex: 0 2 4 ...)
    Args:
        atoms_ucell: ASE atoms for the unitcell
        atomic_mass: Atomic mass for each atom in unitcell
        dof (int): Degrees of freedom
        eigenvecs (array): eigenvectors
        freqs (array): Frequency array in shape of (q, I)
        inverse_mapping (list):  The mapping for crystal to dimer (c1 -> d#, c2 -> d# ...), this will be used to identify atom is in dimer 1 or 2
        n_atoms_xyz (int): number of atoms in dimer
        n_modes (int) : Total number of phonon modes
        n_qpts (int): Number of qpts
        t_vecs (list): Translation vectors
    Return:
        X_allq: X matrix 
        Q_allq: Q vectors
        freq_list (list): The frequencies list in order of Iq index
    """
    select_atoms_input = input("Enter the index of atoms in the unitcell that will be calculated (Press enter will calculated all atoms (When it's not translation dimer))\n"
                               "If translation dimer, enter the unique unit-cell atom indices corresponding to one monomer (ex: 0 1 2 3 0 1 2 3, enter: 0 1 2 3)").strip()
    if select_atoms_input == "":
        select_atoms = []  # means all atoms
    elif "-" in select_atoms_input: # Ex: 0-3 
        start, end = map(int, select_atoms_input.split("-"))
        base_list = list(range(start, end + 1))
        select_atoms = base_list + base_list  # duplicate for dimer
    else: # Ex: 0 1 2 3
        base_list = list(map(int, select_atoms_input.split()))
        select_atoms = base_list + base_list  # duplicate for dimer
  
    n_Iq = n_modes * n_qpts # Number of modes * numer of q-points
    l_v = np.ones(shape=(1,dof)) # left row 1 vector
    r_v = np.ones(shape=(dof,1)) # right column 1 vector
    X_allq = np.zeros((dof, n_Iq), dtype=np.complex128)
    Q_allq = np.zeros(n_Iq, dtype=np.float32) # it is an array because matrix takes large memory
    freq_list = []
    Iq = 0 # the index for Iq
    
    if len(select_atoms) > 0:
        new_atoms = atoms_ucell[select_atoms] # Reformulate unitcell atoms by user selection atoms
        new_atomic_mass = new_atoms.get_masses()
        for I in range(n_modes): # loop over phonon modes index 
            for q in range(n_qpts): # loop over q-points index q
                freq = freqs[q][I]
                freq_list.append(freq)
                Q2 = 1 / np.tanh((planck_J*freq*1e12) / (2*boltzmann_J*temp)) # coth(hbar*omega/2kT)
                Q_allq[Iq] = np.sqrt(Q2) # diagonal matrix for sqrt(coth(...))
                for i, k in enumerate(select_atoms): # loop over atomic index i, k is the index in unitcell 
                    am = new_atomic_mass[i] * mass_au_to_kg # atomic mass
                    pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                    check_map = inverse_mapping[i]
                    if check_map < n_atoms_xyz / 2: # Atoms in 1st dimer (no translation)
                        for ll in (0, 1, 2): # loop over x,y,z directions
                            phase = 1
                            eps = eigenvecs[q, k * 3 + ll, I]
                            uq = eps * pref * phase / np.sqrt(n_qpts)
                            X_allq[i * 3 + ll, Iq] = uq 
                    
                    elif check_map >= n_atoms_xyz / 2: # Atoms in 2nd dimer (from user definition)
                        phase = np.exp(-2j*np.pi * np.dot(qpts[q], t_vecs))
                        for ll in (0, 1, 2): # loop over x,y,z directions
                            eps = eigenvecs[q, k * 3 + ll, I]
                            uq = eps * pref * phase / np.sqrt(n_qpts)
                            X_allq[i * 3 + ll, Iq] = uq 
                Iq += 1
    else:
        n_atoms_ucell = len(atoms_ucell)
        for I in range(n_modes): # loop over phonon modes index 
            for q in range(n_qpts): # loop over q-points index q
                freq = freqs[q][I]
                freq_list.append(freq)
                Q2 = 1 / np.tanh((planck_J*freq*1e12) / (2*boltzmann_J*temp)) # coth(hbar*omega/2kT)
                Q_allq[Iq] = np.sqrt(Q2) # diagonal matrix for sqrt(coth(...))
                for k in range(n_atoms_ucell): # loop over atomic index k
                    am = atomic_mass[k] * mass_au_to_kg # atomic mass
                    pref = np.sqrt(reduced_planck_J / (4*np.pi*am*freq*1e12)) * 1e10 # from meter to angstrom
                    check_map = inverse_mapping[k]
                    if check_map < n_atoms_xyz / 2: # Atoms in 1st dimer (no translation)
                        for ll in (0, 1, 2): # loop over x,y,z directions
                            phase = 1
                            eps = eigenvecs[q, k * 3 + ll, I]
                            uq = eps * pref * phase / np.sqrt(n_qpts)
                            X_allq[k * 3 + ll, Iq] = uq 
                    
                    elif check_map >= n_atoms_xyz / 2: # Atoms in 2nd dimer (from user definition)
                        phase = np.exp(-2j*np.pi * np.dot(qpts[q], t_vecs))
                        for ll in (0, 1, 2): # loop over x,y,z directions
                            eps = eigenvecs[q, k * 3 + ll, I]
                            uq = eps * pref * phase / np.sqrt(n_qpts)
                            X_allq[k * 3 + ll, Iq] = uq 
                Iq += 1

    print(f'X matrix: {X_allq.shape}; Q vectors: {Q_allq.shape}\n'
          'Q should be the matrix but to save the memory vectors are efficient')
    return X_allq, Q_allq, freq_list

In [12]:
def run_SVD(dimer_label, dof, delJ_matrix, freq_list, n_modes, X_allq, Q_allq, p):
    """ Run (1) numpy svd function to obtain 3 matrix (2) obtain U tilde matrix and save the projected mode atomic motions
    (3) V dag tilde matrix and population from the orginal phonon modes
    Args: 
        dimer_label (str): a, b, c ...
        dof (int): Degrees of freedom
        delJ_matrix: The matrix of numerical derivative of transfer integral J 
        freq_list (list): The frequencies list in order of Iq index
        n_modes (int): Number of phonon modes
        X_allq: X matrix 
        Q_allq: Q vectors
        p (int): Number of primary modes 
    Return:
        S (array): Singular value array
        var_svd: SVD variance
        var_p: SVD primary mode variance
        var_b: SVD bath mode variance
        var_svd_plot_list: A list saves variance of each SVD projected phonon mode
        population_p: population matrix for primary mode, it consists of the number from each originl phonon.
        f_sys (list): Primary mode frequencies
    """
    print("--- Run SVD projection ---")
    l_v = np.ones(shape=(1,dof)) # left row 1 vector
    r_v = np.ones(shape=(dof,1)) # right column 1 vector
    XQ = X_allq * Q_allq[None, :]
    svd_input = delJ_matrix @ XQ
    U, S, V_dag = np.linalg.svd(svd_input, full_matrices=False)
    S_mx = np.diag(S) # Diagonal matrix of singular values
    var_svd = (l_v @ U @ S_mx**2 @ U.conj().T @ r_v).real # Use SVD to calculate variance again
    U_p = U[:, :p] # U primary
    S_p = np.diag(S[:p]) # S primary
    V_dag_p = V_dag[:p, :]  # V dagger primary
    U_b = U[:, p:] # U bath
    S_b = np.diag(S[p:]) # S bath
    V_dag_b = V_dag[p:, :] # V dagger bath
    
    print(f'U matrix shape: {U.shape}\n'
          f'Number of singular value: {S.shape}\n'
          f'V dagger matrix shape: {V_dag.shape}')

    var_svd_plot_list = [] # SVD variance per projected mode
    for I in range(0, n_modes): 
        var_svd_plot = (l_v @ U[:, I]) * S[I]**2 * (U[:, I].conj().T @ r_v)
        var_svd_plot_list.append(var_svd_plot[0].real)
    
    var_p = (l_v @ U_p @ S_p**2 @ U_p.conj().T @ r_v).real # Variance from projected primary modes
    var_p = var_p[0][0]
    var_b = (l_v @ U_b @ S_b**2 @ U_b.conj().T @ r_v).real # Variance from projected bath modes
    var_b = var_b[0][0]
    
    U_tilt = np.linalg.inv(delJ_matrix) @ U # Create U tilde matrix: the matrix to describe atomic motion by SVD (you should pick column)
    for ip in range(p): # First p primary mode motion and save files as disp_{dimer}_{p}.txt
        disp = U_tilt[:,ip].real
        disp_cart = disp.reshape(-1,3)
        np.savetxt(f"disp/disp_{dimer_label}_{ip+1}.txt", disp_cart)

    V_dag_tilt = V_dag[:p, :] / Q_allq[None, :] # Create V_dag tilde matrix: the matrix to describe the SVD projection (you should pick row)
    population_p = np.zeros((p, n_modes),dtype=float) # The contribution from each mode. For each mode, we sum up the population from each wavevector q
    for ip in range(p):
        iq = 0
        for I in range(n_modes):
            value = 0
            for q in range(n_qpts):
                value += V_dag_tilt[ip, iq].real
                iq += 1
            population_p[ip, I] = value

    freq_2 = np.array(freq_list)**2 #Calculate the projected mode frequencies. Derivation is in SI
    H_sys = (V_dag_tilt * freq_2[None,:]) @ V_dag_tilt.conj().T 
    eigvals, eigvecs = np.linalg.eigh(H_sys)
    f_sys = np.sqrt(np.clip(eigvals.real, 0, None))

    print("--- Finish SVD projection ---")
    return S, var_svd, var_p, var_b, var_svd_plot_list, population_p, f_sys

In [13]:
def calc_variance(dof, delJ_matrix, X_allq, Q_allq, n_modes, n_qpts):
    """ Variance calculations. 
    Args:
        dof (int): Degrees of freedom
        delJ_matrix: The matrix of numerical derivative of transfer integral J 
        X_allq: X matrix 
        Q_allq: Q vectors
        n_modes (int) : Total number of phonon modes
        n_qpts (int): Number of qpts
    Return:
        var_total: Total variance from original phonons
        var_a: Variance from acoustic phonons
        var_o: Variance from optical phonons
        var_plot_mode_list: A list saves variance for each original phonons
        var_plot_wn_list: A list saves variance for each original phonons based on wavenumber
    """
    print("--- Calculate variance of transfer integral ---")
    l_v = np.ones(shape=(1,dof)) # left row 1 vector
    r_v = np.ones(shape=(dof,1)) # right column 1 vector
    XQ = X_allq * Q_allq[None, :]
    var_total = l_v @ delJ_matrix @ (XQ @ XQ.conj().T).real @ delJ_matrix.T @ r_v # Variance is calculated here
    var_a = l_v @ delJ_matrix @ (XQ[:,0:3*n_qpts] @ XQ[:,0:3*n_qpts].conj().T).real @ delJ_matrix.T @ r_v # Variance from acoustic modes
    var_o = l_v @ delJ_matrix @ (XQ[:,3*n_qpts:] @ XQ[:,3*n_qpts:].conj().T).real @ delJ_matrix.T @ r_v # Variance from optical modes

    var_plot_mode_list = [] # variance vs mode index
    var_plot_wn_list = [] # variance vs mode wavenumber
    iq = 0
    for I in range(0, n_modes):
        var_plot_mode = 0
        for q in range(n_qpts):
            value = l_v @ delJ_matrix @ XQ[:,iq]
            var = value @ value.conj()
            var_plot_mode += var.real
            var_plot_wn_list.append(var.real)
            iq += 1
        var_plot_mode_list.append(var_plot_mode)

    return var_total, var_a, var_o, var_plot_mode_list, var_plot_wn_list

In [14]:
print("This notebook is to calculate variance of transfer integral and perform SVD projection\n"
      "Code Explanation:\n"
      "Del-J matrix provides the change of transfer integral per unit atomic displacement ...\n"
      "U matrix is the prefactor for the distance; Q matrix is thermal contirbution (coth)\n"
      "--- The user will have to fill up the input parameter below ---")

# ------------------ Input parameter --------------------- #
temp = 300 # TEMPERATURE: Kelvin
mesh = [8,8,8] # phonopy mesh grid
p = 8 # choose rank for primary modes
dimer_label = 'a' # label for dimer
monomer1 = 1 # label for monomer 1
monomer2 = 2 # label for monomer 2
molecule = 'PEN' # Will be used on naming for the plots
crystal = 'CONTCAR' # The unitcell file for phonopy simulation
xyz = 'dimer_A.xyz' # dimer xyz file
n_atoms_xyz = 72 # The number of atoms in dimer file (first line in .xyz file)
dof = n_atoms_xyz * 3 # Degrees of freedom in dimer
# Note for the phase exp(-iqR):
# Monomer 1 must in the unitcell, depends on the unitcel. It can be 1 molecule or 2 molecules in the unitcell,
# Different dimer can also have different phase
# phase_x = np.exp(-2j*np.pi * qpts[q][0] * cell_vector) Because cell_vector is 0 -> The phase is 1

print("------------------------------------------\n"
       "Input parameters:\n"
       f"Temperature (Kelvin): {temp}\n"
       f"Mesh points for phonopy: {mesh}\n"
       f"Number of primary modes: {p}\n"
       f"Dimer: {dimer_label}; monomer1: {monomer1}; monomer2: {monomer2}\n"
       f"Material: {molecule}\n"
       f"Crystal structure file: {crystal}\n"
       f"Dimer xyz: {xyz}\n"
       f"Number of atoms in dimer: {n_atoms_xyz}\n"
       f"Degrees of freedom: {dof}\n"
       "------------------------------------------")
# ------------------ End of Input parameter --------------------- #

This notebook is to calculate variance of transfer integral and perform SVD projection
Code Explanation:
Del-J matrix provides the change of transfer integral per unit atomic displacement ...
U matrix is the prefactor for the distance; Q matrix is thermal contirbution (coth)
--- The user will have to fill up the input parameter below ---
------------------------------------------
Input parameters:
Temperature (Kelvin): 300
Mesh points for phonopy: [8, 8, 8]
Number of primary modes: 8
Dimer: a; monomer1: 1; monomer2: 2
Material: PEN
Crystal structure file: CONTCAR
Dimer xyz: dimer_A.xyz
Number of atoms in dimer: 72
Degrees of freedom: 216
------------------------------------------


In [ ]:
# ---------- Call the functions here --------------# 
atoms_ucell, atomic_mass, n_atoms_ucell, n_modes, t_vecs, translation_dimer = ucell_info(crystal, monomer1, monomer2)
eigenvecs, freqs, freqs_flatten, n_qpts, qpts = phonon()
inverse_mapping = mapping_atoms(dimer_label, translation_dimer, xyz)
delJ_matrix = make_dJ_matrix(dimer_label, n_atoms_xyz, translation_dimer)
X_allq, Q_allq, freq_list = make_XQ_matrix(atoms_ucell, atomic_mass, dof, eigenvecs, freqs, inverse_mapping, n_atoms_xyz, n_modes, n_qpts, t_vecs)
S, var_svd, var_p, var_b, var_svd_plot_list, population_p, f_sys = run_SVD(dimer_label, dof, delJ_matrix, freq_list, n_modes, X_allq, Q_allq, p)
var_total, var_a, var_o, var_plot_mode_list, var_plot_wn_list = calc_variance(dof, delJ_matrix, X_allq, Q_allq, n_modes, n_qpts)

--- Lattice translational vectors: [0.5 0.5 0. ]
If the index not the integer, it is still in the unitcell. It should be 0 0 0
Please define translation vectors by yourself!!! ---


Enter numbers separated by space:  0 0 0


--- If the translation vectors is not zero instead is an integer at any direction, select y!!! ---


This dimer is formed by tranlsation molecules (y or n) n


--- Solving Hessian Matrix (second order derivative of potential energy) by phonopy ---
--- Check whether there is any negative frequencies and remove them ---
--- Removing 0 q-points containing imaginary modes ---
Filtered frequency shape: (256, 216)
Number of remaining q-points = 256
Eigenvectors shape (qpts, dof, mode) = (256, 216, 216)
--- Finishing Solving Hessian Matrix ---
--- Atomic index mapping (dimer # : crystal #): Ex: d1 -> n; d2 -> m ...
[57 63  9 66 68 48  2 13 14 49 19 23 26 28 34 39 41  3 51 12 54 15 56 18
 22 62 27 67 29 69 35 38 40  8 50 55 24 65 30 64 33 36 43 44 47 52  1 58
  4 61  6 11 71 17 20 25 31 32 37 42 45 46 53 59 60 70  0  5  7 10 16 21] ---
--- Finishing Mapping and Writing dimer_map.xyz file to the folder ---
--- Generate delta J matrix ---
--- Obtain delta J matrix, now start to map dJ ---
--- Finish generating delta J matrix 
Delta J matrix shape: (216, 216) ---


Enter the index of atoms in the unitcell that will be calculated (Press enter will calculated all atoms (When it's not translation dimer))
If translation dimer, enter the unique unit-cell atom indices corresponding to one monomer (ex: 0 1 2 3 0 1 2 3, enter: 0 1 2 3) 


In [ ]:
# ----------- Output parameters -------------- #
print("Output parameters:\n"
      f"1. Number of bath modes: {n_modes - p}\n"
      f'4. The first 10 singular values: {S[0:10]}\n'
      f'5. The variance by acoustic (US^2U^T): {np.round(var_a[0][0],6)}\n'
      f'5. The variance by optical (US^2U^T): {np.round(var_o[0][0],6)}\n'
      f'6. Variance from primary modes: {np.round(var_p,6)}\n'
      f'8. Variance from bath modes: {np.round(var_b,6)}\n'
      f'10. Variance from primary + bath modes: {np.round((var_p + var_b),6)}\n'
      f'11. Variance of the system modes: {np.round(var_total[0][0],6)}\n'
      f'12. Percentage of variance of primary modes to the system modes: {np.round((var_p / var_total[0][0]),4) * 100}\n'
      f'13. With SVD mode projection, system modes: {np.round(f_sys*thz_to_wnumber,1)} cm^-1')
# ----------- End of Output parameters -------------- #

In [ ]:
while True:
    I = input('Please input the mode index of phonon modes, or q to quit').strip().lower()
    if I == 'q':
        break
    freq_mode_min, freq_mode_max, freq_mode_mean = freq_summary_for_mode(freqs, int(I))
    print(f'Min freq {freq_mode_min:.2f} cm^-1, Max freq {freq_mode_max:.2f} cm^-1, Mean freq {freq_mode_mean:.2f} cm^-1')
    save_motion = input('Please input the y or n to decide whether save atomic motion or not').strip().lower()
    if save_motion == 'y':
        write_original_motion(atoms_ucell, atomic_mass, dimer_label, dof, eigenvecs, freqs, inverse_mapping, int(I), n_atoms_xyz, n_qpts, t_vecs)
    else:
        continue

In [ ]:
# --------------- Make figures below -------------------- #

In [ ]:
color = ['lime', 'deeppink', 'chocolate', 'blue', 'deepskyblue', 'orange', 'red', 'deepskyblue', 'olive', 'gray', 'purple']
fig, axes = plt.subplots(p, 1, figsize=(12,8), sharex=True)
axes[-1].set_xlabel('Original Phonon Mode Index')
axes[1].set_ylabel('Population')
axes[0].set_title(f'{molecule} Dimer {dimer_label} SVD Population')

x = np.arange(1, n_modes+1)
nticks = 8
tick_pos = np.linspace(1, n_modes, nticks, dtype=int)
#ax_top.set_xticks(tick_pos)
#ax_top.set_xticklabels([f"{freq_optical_plot[i-1]:.1f}" for i in tick_pos])
#ax_top.set_xlabel("Frequency (cm$^{-1}$)")
for i in range(p):
    axes[i].bar(x, population_p[i], color=color[i], edgecolor='black', linewidth=0.35, label=f'#{i+1} Primary Mode')
    axes[i].set_ylim(-0.7, 0.7)
    axes[i].set_xlim(0, 220)
for ax in axes:
    ax.legend(frameon=False, fontsize='small')

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.set_xlabel(r'Frequency $(cm^{-1})$')
ax.set_ylabel(r'Variance $\sigma^2 (meV^2)$')
ax.set_title(f'{molecule} Dimer {dimer_label} SVD Variance vs Frequencies')
ax.set_xlim(0,200)
ax.set_ylim(0, 0.0005)

ax.bar(freqs_flatten[0:3*n_qpts]*thz_to_wnumber, var_plot_wn_list[:3*n_qpts], color='hotpink', label='Original Acoustic Modes')
ax.bar(freqs_flatten[3*n_qpts:5200]*thz_to_wnumber, var_plot_wn_list[3*n_qpts:5200], color='blue', label='Original Optical Modes')
ax.bar(f_sys*thz_to_wnumber, var_svd_plot_list[:p], color='green', label='Projected Modes')
plt.legend(frameon=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.set_xlabel('Projected/Original Phonon Mode Index')
ax.set_ylabel(r'Variance $\sigma^2$')
ax.set_ylabel(r'Variance $\sigma^2$')
ax.set_title(f'{molecule} Dimer {dimer_label} SVD Variance vs Mode Index')
ax.set_xlim(0,50)
ax.set_ylim(0, 0.0008)
#ax.vlines(p,0,0.01, linestyle='--', color='gray')

x_a = np.arange(1, 4)
x_o = np.arange(4, n_modes+1)
x_svd = np.arange(1, n_modes+1)
ax.plot(x_a, var_plot_mode_list[0:3], linewidth=0.8, color='hotpink', marker='.', label='Original Acoustic Modes')
ax.plot(x_o, var_plot_mode_list[3:], linewidth=0.8, color='aqua', marker='.', label='Original Optical Modes')
ax.plot(x_svd, var_svd_plot_list, linewidth=0.8, color='springgreen', marker='.', label='Projected Modes')
plt.legend(frameon=False)